Install required libraries

In [1]:
!pip install -q transformers sentence-transformers language-tool-python nltk rouge-score sentencepiece

Import Libraries

In [2]:
import re
import string
import warnings

import torch
import nltk

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

from rouge_score import rouge_scorer

import language_tool_python

warnings.filterwarnings("ignore")

print("Libraries imported successfully.")

Libraries imported successfully.


Download NLTK resources

In [3]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

print("NLTK resources downloaded.")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Ranjana\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Ranjana\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ranjana\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


NLTK resources downloaded.


Check hardware

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

Device: cpu
Running on CPU


Load the T5 paraphrasing model

In [5]:
MODEL_NAME = "Vamsi/T5_Paraphrase_Paws"

print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading model...")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

model = model.to(device)

print("T5 paraphrasing model loaded successfully.")

Loading tokenizer...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/1.79k [00:00<?, ?B/s]

Loading model...


model.safetensors: reconstructing file:   0%|          |  0.00B /  892MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

T5 paraphrasing model loaded successfully.


Load semantic similarity model

In [6]:
SIMILARITY_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

similarity_model = SentenceTransformer(SIMILARITY_MODEL)

print("Semantic similarity model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic similarity model loaded successfully.


Load grammer and spelling checker

In [7]:
try:
    grammar_tool = language_tool_python.LanguageTool("en-US")
    print("LanguageTool initialized successfully.")

except Exception as e:
    grammar_tool = None
    print("LanguageTool could not be initialized.")
    print("Error:", e)

LanguageTool initialized successfully.


Text preprocessing function

In [8]:
def preprocess_text(text):
    """
    Basic text preprocessing.

    Parameters:
        text (str): Input text

    Returns:
        str: Cleaned text
    """

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing whitespace
    text = text.strip()

    return text

Sentence splitting

In [9]:
def split_into_sentences(text):
    """
    Splits a block of text into sentences.
    """

    sentences = nltk.sent_tokenize(text)

    return sentences

Test preprocessing

In [10]:
sample_text = """
Machine learning is a branch of artificial intelligence.
It enables computers to learn from data without being explicitly programmed.
"""

clean_text = preprocess_text(sample_text)

print("Cleaned text:")
print(clean_text)

print("\nSentences:")
for i, sentence in enumerate(split_into_sentences(clean_text), 1):
    print(f"{i}. {sentence}")

Cleaned text:
Machine learning is a branch of artificial intelligence. It enables computers to learn from data without being explicitly programmed.

Sentences:
1. Machine learning is a branch of artificial intelligence.
2. It enables computers to learn from data without being explicitly programmed.


T5 paraphrasing function

In [11]:
def paraphrase_sentence(
    sentence,
    num_return_sequences=1,
    num_beams=5,
    temperature=1.0,
    max_length=128
):
    """
    Generates paraphrases using T5.

    Parameters:
        sentence: Input sentence
        num_return_sequences: Number of paraphrases
        num_beams: Beam search size
        temperature: Sampling temperature
        max_length: Maximum generated length

    Returns:
        List of paraphrased sentences
    """

    sentence = preprocess_text(sentence)

    # T5 paraphrasing format
    input_text = "paraphrase: " + sentence + " </s>"

    encoded = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():

        outputs = model.generate(
            **encoded,
            max_length=max_length,
            num_beams=num_beams,
            num_return_sequences=num_return_sequences,
            temperature=temperature,
            early_stopping=True
        )

    paraphrases = []

    for output in outputs:

        decoded = tokenizer.decode(
            output,
            skip_special_tokens=True
        )

        paraphrases.append(decoded.strip())

    return paraphrases

Test the paraphrasing model

In [12]:
text = """
Machine learning is a branch of artificial intelligence that enables computers
to learn from data without being explicitly programmed.
"""

paraphrases = paraphrase_sentence(text)

print("Original:")
print(text)

print("\nParaphrased:")
print(paraphrases[0])

Original:

Machine learning is a branch of artificial intelligence that enables computers
to learn from data without being explicitly programmed.


Paraphrased:
Machine learning is a branch of artificial intelligence that enables computers to learn from data without being explicitly programmed .


Generate multiple paraphrases

In [13]:
def generate_multiple_paraphrases(sentence, n=5):

    sentence = preprocess_text(sentence)

    input_text = "paraphrase: " + sentence + " </s>"

    encoded = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():

        outputs = model.generate(
            **encoded,
            max_length=128,
            num_beams=10,
            num_return_sequences=n,
            no_repeat_ngram_size=2,
            early_stopping=True
        )

    results = []

    for output in outputs:

        decoded = tokenizer.decode(
            output,
            skip_special_tokens=True
        )

        results.append(decoded.strip())

    return results

Generate five paraphrases

In [14]:
input_sentence = """
Artificial intelligence is transforming many industries by automating
complex tasks and helping organizations make better decisions.
"""

results = generate_multiple_paraphrases(input_sentence, n=5)

print("ORIGINAL:")
print(input_sentence)

print("\nGENERATED PARAPHRASES:\n")

for i, result in enumerate(results, 1):

    print(f"{i}. {result}")

ORIGINAL:

Artificial intelligence is transforming many industries by automating
complex tasks and helping organizations make better decisions.


GENERATED PARAPHRASES:

1. Artificial intelligence is transforming many industries by automating complex tasks and helping organizations make better decisions .
2. Artificial intelligence is transforming many industries by automating complex tasks and helping organizations to make better decisions .
3. Artificial intelligence transforms many industries by automating complex tasks and helping organizations make better decisions .
4. Artificial intelligence is changing many industries by automating complex tasks and helping organizations make better decisions .
5. Artificial intelligence is changing many industries by automating complex tasks and helping organizations to make better decisions .


Paraphrase an entire block of text

In [15]:
def paraphrase_text(text):

    """
    Paraphrases a complete block of text sentence by sentence.
    """

    text = preprocess_text(text)

    sentences = split_into_sentences(text)

    paraphrased_sentences = []

    for sentence in sentences:

        result = paraphrase_sentence(
            sentence,
            num_return_sequences=1
        )[0]

        paraphrased_sentences.append(result)

    return " ".join(paraphrased_sentences)

Test complete paragraph paraphrasing





In [16]:
input_text = """
Machine learning is an important field of artificial intelligence.
It allows computers to identify patterns in data and make predictions.
Machine learning is widely used in healthcare, finance, education, and transportation.
"""

output_text = paraphrase_text(input_text)

print("========== ORIGINAL TEXT ==========\n")
print(input_text)

print("\n========== PARAPHRASED TEXT ==========\n")
print(output_text)

========== ORIGINAL TEXT ==========


Machine learning is an important field of artificial intelligence.
It allows computers to identify patterns in data and make predictions.
Machine learning is widely used in healthcare, finance, education, and transportation.


========== PARAPHRASED TEXT ==========

Machine learning is an important field of artificial intelligence . It allows computers to identify patterns in data and make predictions . Machine learning is widely used in healthcare, finance, education and transportation .


Semantic similarity

In [17]:
def semantic_similarity(original, paraphrased):
    """
    Calculates cosine similarity between original and paraphrased text.
    """

    embeddings = similarity_model.encode(
        [original, paraphrased],
        convert_to_tensor=True
    )

    similarity = cos_sim(
        embeddings[0],
        embeddings[1]
    )

    return float(similarity.item())

Test semantic similarity

In [18]:
original = """
Machine learning allows computers to learn patterns from data.
"""

paraphrased = """
Computers can identify patterns in data through machine learning.
"""

score = semantic_similarity(
    original,
    paraphrased
)

print("Semantic Similarity:", round(score, 4))

Semantic Similarity: 0.89


Interpretation

A value closer to 1.0 means the two texts are semantically more similar.

Calculate lexical originality

In [19]:
def tokenize_words(text):

    text = text.lower()

    words = re.findall(r"\b[a-zA-Z]+\b", text)

    return words


def lexical_overlap(original, paraphrased):

    original_words = set(tokenize_words(original))
    paraphrased_words = set(tokenize_words(paraphrased))

    if len(original_words) == 0:
        return 0

    common_words = original_words.intersection(
        paraphrased_words
    )

    overlap = len(common_words) / len(original_words)

    return overlap


def originality_score(original, paraphrased):

    overlap = lexical_overlap(
        original,
        paraphrased
    )

    originality = (1 - overlap) * 100

    return originality

Test originality

In [20]:
original = """
Machine learning enables computers to learn from historical data.
"""

paraphrased = """
Computers can discover patterns from previous information using machine learning.
"""

overlap = lexical_overlap(
    original,
    paraphrased
)

originality = originality_score(
    original,
    paraphrased
)

print("Lexical overlap:", round(overlap, 4))

print(
    "Estimated originality:",
    round(originality, 2),
    "%"
)

Lexical overlap: 0.4444
Estimated originality: 55.56 %


Grammer and spelling checking

In [21]:
def grammar_check(text):
    """
    Checks grammar, spelling, punctuation, and style
    using LanguageTool.
    """

    if grammar_tool is None:
        return {
            "error_count": None,
            "errors": []
        }

    matches = grammar_tool.check(text)

    errors = []

    for match in matches:

        # Get error length safely
        error_length = getattr(
            match,
            "errorLength",
            getattr(match, "error_length", 0)
        )

        errors.append({
            "message": match.message,
            "offset": match.offset,
            "length": error_length,
            "category": match.category
        })

    return {
        "error_count": len(errors),
        "errors": errors
    }

Test grammer checking

In [22]:
test_text = """
Machine learning are very useful technology.
It help computers understand data.
"""

grammar_result = grammar_check(test_text)

print("Number of detected issues:",
      grammar_result["error_count"])

print("\nDetected issues:")

for error in grammar_result["errors"]:

    print(
        "-",
        error["message"]
    )

Number of detected issues: 1

Detected issues:
- After ‘It’, use the third-person verb form “helps”.


Grammer score

In [23]:
def grammar_score(text):

    result = grammar_check(text)

    if result["error_count"] is None:
        return None

    words = tokenize_words(text)

    if len(words) == 0:
        return 0

    errors_per_100_words = (
        result["error_count"] / len(words)
    ) * 100

    score = max(
        0,
        100 - errors_per_100_words * 10
    )

    return round(score, 2)

Test grammer score

In [24]:
text = """
Machine learning is a powerful technology.
It helps computers learn from data.
"""

score = grammar_score(text)

print("Grammar/fluency score:", score, "/ 100")

Grammar/fluency score: 100.0 / 100


BLEU evaluation

In [25]:
def bleu_score(reference, candidate):

    reference_tokens = tokenize_words(reference)
    candidate_tokens = tokenize_words(candidate)

    smoothie = SmoothingFunction().method4

    score = sentence_bleu(
        [reference_tokens],
        candidate_tokens,
        weights=(0.25, 0.25, 0.25, 0.25),
        smoothing_function=smoothie
    )

    return score

Test BLEU

In [26]:
reference = """
Computers learn patterns from data through machine learning.
"""

candidate = """
Machine learning allows computers to identify patterns in data.
"""

score = bleu_score(
    reference,
    candidate
)

print("BLEU Score:", round(score, 4))

BLEU Score: 0.0795


ROUGE evaluation

In [27]:
def rouge_scores(reference, candidate):

    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=True
    )

    scores = scorer.score(
        reference,
        candidate
    )

    return {
        "ROUGE-1": scores["rouge1"].fmeasure,
        "ROUGE-2": scores["rouge2"].fmeasure,
        "ROUGE-L": scores["rougeL"].fmeasure
    }

Test ROUGE

In [28]:
reference = """
Computers learn patterns from data through machine learning.
"""

candidate = """
Machine learning allows computers to identify patterns in data.
"""

scores = rouge_scores(
    reference,
    candidate
)

for metric, score in scores.items():

    print(
        metric,
        ":",
        round(score, 4)
    )

ROUGE-1 : 0.5882
ROUGE-2 : 0.1333
ROUGE-L : 0.3529


Complete evaluation function

In [29]:
def evaluate_paraphrase(
    original,
    paraphrased,
    reference=None
):

    semantic = semantic_similarity(
        original,
        paraphrased
    )

    overlap = lexical_overlap(
        original,
        paraphrased
    )

    originality = originality_score(
        original,
        paraphrased
    )

    grammar = grammar_score(
        paraphrased
    )

    evaluation = {

        "Semantic Similarity": round(
            semantic, 4
        ),

        "Lexical Overlap": round(
            overlap, 4
        ),

        "Originality (%)": round(
            originality, 2
        ),

        "Grammar Score": grammar
    }

    # BLEU and ROUGE are calculated
    # only when a reference paraphrase exists.

    if reference is not None:

        evaluation["BLEU"] = round(
            bleu_score(
                reference,
                paraphrased
            ),
            4
        )

        rouge = rouge_scores(
            reference,
            paraphrased
        )

        evaluation.update({
            key: round(value, 4)
            for key, value in rouge.items()
        })

    return evaluation

Run complete evaluation

In [30]:
original = """
Machine learning is a branch of artificial intelligence that
allows computers to learn from data without explicit programming.
"""

paraphrased = """
Machine learning is an area of artificial intelligence in which
computers learn from data instead of relying on direct programming.
"""

evaluation = evaluate_paraphrase(
    original,
    paraphrased
)

print("========== EVALUATION ==========\n")

for metric, value in evaluation.items():

    print(
        f"{metric}: {value}"
    )

========== EVALUATION ==========

Semantic Similarity: 0.9068
Lexical Overlap: 0.6111
Originality (%): 38.89
Grammar Score: 100.0


Ovaerall quality assessment

In [31]:
def overall_quality_score(evaluation):

    semantic = evaluation["Semantic Similarity"]

    originality = evaluation["Originality (%)"] / 100

    grammar = evaluation["Grammar Score"]

    if grammar is None:
        grammar = 0

    grammar = grammar / 100

    # Weighted score
    #
    # Semantic preservation = 50%
    # Originality = 25%
    # Grammar/fluency = 25%

    score = (
        semantic * 0.50 +
        originality * 0.25 +
        grammar * 0.25
    ) * 100

    return round(score, 2)

Test overall score

In [32]:
score = overall_quality_score(
    evaluation
)

print(
    "Overall Paraphrasing Quality Score:",
    score,
    "/ 100"
)

Overall Paraphrasing Quality Score: 80.06 / 100


Complete paraphrasing pipeline

In [33]:
def paraphrase_pipeline(text):

    print("=" * 70)
    print("AI-POWERED PARAPHRASING TOOL")
    print("=" * 70)

    # Step 1: Preprocess
    text = preprocess_text(text)

    # Step 2: Display original
    print("\nORIGINAL TEXT:")
    print(text)

    # Step 3: Generate paraphrase
    print("\nGenerating paraphrase...")

    paraphrased = paraphrase_text(text)

    # Step 4: Display output
    print("\nPARAPHRASED TEXT:")
    print(paraphrased)

    # Step 5: Evaluate
    print("\nEvaluating output...")

    evaluation = evaluate_paraphrase(
        text,
        paraphrased
    )

    print("\nEVALUATION RESULTS:")
    print("-" * 40)

    for metric, value in evaluation.items():

        print(
            f"{metric}: {value}"
        )

    # Step 6: Overall score
    overall_score = overall_quality_score(
        evaluation
    )

    print(
        "\nOverall Quality Score:",
        overall_score,
        "/ 100"
    )

    return paraphrased, evaluation

Run the complete application

In [34]:
input_text = """
Artificial intelligence has become an important technology
in modern society. It is used in healthcare, finance,
education, transportation, and many other industries.
AI systems can analyze large amounts of data and assist
people in making better decisions.
"""

paraphrased_text, evaluation = paraphrase_pipeline(
    input_text
)

AI-POWERED PARAPHRASING TOOL

ORIGINAL TEXT:
Artificial intelligence has become an important technology in modern society. It is used in healthcare, finance, education, transportation, and many other industries. AI systems can analyze large amounts of data and assist people in making better decisions.

Generating paraphrase...

PARAPHRASED TEXT:
Artificial intelligence has become an important technology in modern society . It is used in healthcare, finance, education, transportation and many other industries. AI systems can analyze large amounts of data and assist people in making better decisions .

Evaluating output...

EVALUATION RESULTS:
----------------------------------------
Semantic Similarity: 0.9997
Lexical Overlap: 1.0
Originality (%): 0.0
Grammar Score: 45.95

Overall Quality Score: 61.47 / 100


Console input

In [35]:
print("=" * 70)
print("AI PARAPHRASING TOOL")
print("=" * 70)

print("\nEnter the text you want to paraphrase.")
print("Press ENTER twice when finished.\n")

lines = []

while True:

    line = input()

    if line == "":
        break

    lines.append(line)

user_text = " ".join(lines)

if user_text.strip():

    paraphrased_text, evaluation = paraphrase_pipeline(
        user_text
    )

else:

    print("No input text was provided.")

AI PARAPHRASING TOOL

Enter the text you want to paraphrase.
Press ENTER twice when finished.



 Technology has significantly improved healthcare services. Doctors can use digital medical records, artificial intelligence, and advanced diagnostic tools to identify diseases earlier and provide better treatment to patients.
 


AI-POWERED PARAPHRASING TOOL

ORIGINAL TEXT:
Technology has significantly improved healthcare services. Doctors can use digital medical records, artificial intelligence, and advanced diagnostic tools to identify diseases earlier and provide better treatment to patients.

Generating paraphrase...

PARAPHRASED TEXT:
Technology has significantly improved healthcare services. Doctors can use digital medical records, artificial intelligence and advanced diagnostic tools to identify diseases earlier and provide better treatment to patients .

Evaluating output...

EVALUATION RESULTS:
----------------------------------------
Semantic Similarity: 0.9987
Lexical Overlap: 1.0
Originality (%): 0.0
Grammar Score: 64.29

Overall Quality Score: 66.01 / 100


Multiple candidate selection

In [36]:
def select_best_paraphrase(
    original,
    candidates
):

    scored_candidates = []

    for candidate in candidates:

        similarity = semantic_similarity(
            original,
            candidate
        )

        originality = originality_score(
            original,
            candidate
        ) / 100

        grammar = grammar_score(candidate)

        if grammar is None:
            grammar = 0.5
        else:
            grammar = grammar / 100

        # Candidate selection score
        score = (
            similarity * 0.50 +
            originality * 0.25 +
            grammar * 0.25
        )

        scored_candidates.append(
            (
                candidate,
                score,
                similarity,
                originality,
                grammar
            )
        )

    scored_candidates.sort(
        key=lambda x: x[1],
        reverse=True
    )

    return scored_candidates

Test candidate selection

In [37]:
original = """
Technology has changed the way people communicate with each other.
"""

candidates = generate_multiple_paraphrases(
    original,
    n=5
)

ranked_candidates = select_best_paraphrase(
    original,
    candidates
)

print("ORIGINAL:")
print(original)

print("\nCANDIDATES:\n")

for i, item in enumerate(
    ranked_candidates,
    1
):

    candidate = item[0]
    score = item[1]

    print(
        f"{i}. {candidate}"
    )

    print(
        f"   Quality score: {score:.4f}\n"
    )

ORIGINAL:

Technology has changed the way people communicate with each other.


CANDIDATES:

1. Technology has changed the way people communicate with one another.
   Quality score: 0.7965

2. Technology changed the way people communicate with each other.
   Quality score: 0.7510

3. Technology has changed the way people communicate with each other.
   Quality score: 0.7500

4. The technology has changed the way people communicate with each other.
   Quality score: 0.7432

5. Technology has changed the way people communicate with each other .
   Quality score: 0.5000



Final improved pipeline

In [38]:
def advanced_paraphrase(text):

    text = preprocess_text(text)

    print("=" * 70)
    print("ORIGINAL TEXT")
    print("=" * 70)

    print(text)

    print("\nGenerating multiple candidates...")

    sentences = split_into_sentences(text)

    final_sentences = []

    for sentence in sentences:

        candidates = generate_multiple_paraphrases(
            sentence,
            n=5
        )

        ranked = select_best_paraphrase(
            sentence,
            candidates
        )

        best_candidate = ranked[0][0]

        final_sentences.append(
            best_candidate
        )

    final_text = " ".join(
        final_sentences
    )

    print("\n" + "=" * 70)
    print("FINAL PARAPHRASED TEXT")
    print("=" * 70)

    print(final_text)

    evaluation = evaluate_paraphrase(
        text,
        final_text
    )

    print("\n" + "=" * 70)
    print("EVALUATION")
    print("=" * 70)

    for metric, value in evaluation.items():

        print(
            f"{metric}: {value}"
        )

    return final_text, evaluation

Run advanced system

In [39]:
text = """
Machine learning is widely used in modern applications.
It helps organizations analyze large datasets, identify patterns,
and make predictions. These technologies are increasingly being
used in healthcare, banking, education, and transportation.
"""

final_text, evaluation = advanced_paraphrase(
    text
)

ORIGINAL TEXT
Machine learning is widely used in modern applications. It helps organizations analyze large datasets, identify patterns, and make predictions. These technologies are increasingly being used in healthcare, banking, education, and transportation.

Generating multiple candidates...

FINAL PARAPHRASED TEXT
Machine learning is widely used in modern applications. It helps organizations analyze large datasets, identify patterns and make predictions. These technologies are increasingly used in healthcare, banking, education and transportation.

EVALUATION
Semantic Similarity: 0.9986
Lexical Overlap: 0.9643
Originality (%): 3.57
Grammar Score: 100.0


Evaluation using a reference paraphrase

In [40]:
test_data = [

    {
        "original":
        "Machine learning enables computers to learn from data.",

        "reference":
        "Computers can learn patterns from data using machine learning."
    },

    {
        "original":
        "Artificial intelligence is transforming modern healthcare.",

        "reference":
        "Modern healthcare is being transformed by artificial intelligence."
    },

    {
        "original":
        "Cloud computing allows organizations to access resources over the internet.",

        "reference":
        "Organizations can use internet-based cloud computing to access resources."
    },

    {
        "original":
        "Data science combines statistics, programming, and domain knowledge.",

        "reference":
        "Statistics, programming, and subject expertise are combined in data science."
    },

    {
        "original":
        "Deep learning uses neural networks to learn complex patterns.",

        "reference":
        "Neural networks in deep learning can identify complicated patterns."
    }

]

print("Number of test examples:", len(test_data))

Number of test examples: 5


Evaluate the test dataset

In [41]:
all_results = []

for i, item in enumerate(test_data, 1):

    original = item["original"]
    reference = item["reference"]

    generated = paraphrase_text(
        original
    )

    evaluation = evaluate_paraphrase(
        original,
        generated,
        reference
    )

    evaluation["Example"] = i

    all_results.append(
        evaluation
    )

    print("\n" + "=" * 70)

    print(
        f"Example {i}"
    )

    print("\nOriginal:")
    print(original)

    print("\nGenerated:")
    print(generated)

    print("\nReference:")
    print(reference)

    print("\nEvaluation:")

    for metric, value in evaluation.items():

        if metric != "Example":
            print(
                f"{metric}: {value}"
            )


Example 1

Original:
Machine learning enables computers to learn from data.

Generated:
Machine learning enables computers to learn from data.

Reference:
Computers can learn patterns from data using machine learning.

Evaluation:
Semantic Similarity: 1.0
Lexical Overlap: 1.0
Originality (%): 0.0
Grammar Score: 100.0
BLEU: 0.0984
ROUGE-1: 0.7059
ROUGE-2: 0.2667
ROUGE-L: 0.4706

Example 2

Original:
Artificial intelligence is transforming modern healthcare.

Generated:
Artificial intelligence is transforming modern healthcare.

Reference:
Modern healthcare is being transformed by artificial intelligence.

Evaluation:
Semantic Similarity: 1.0
Lexical Overlap: 1.0
Originality (%): 0.0
Grammar Score: 100.0
BLEU: 0.1041
ROUGE-1: 0.8571
ROUGE-2: 0.3333
ROUGE-L: 0.2857

Example 3

Original:
Cloud computing allows organizations to access resources over the internet.

Generated:
Cloud computing allows organizations to access resources over the internet .

Reference:
Organizations can use inter

Convert evaluation results to DataFrame

In [42]:
import pandas as pd

results_df = pd.DataFrame(
    all_results
)

results_df

,Semantic Similarity,Lexical Overlap,Originality (%),Grammar Score,BLEU,ROUGE-1,ROUGE-2,ROUGE-L,Example
0,1.0000,1.0,0.0,100.0,0.0984,0.7059,0.2667,0.4706,1
1,1.0000,1.0,0.0,100.0,0.1041,0.8571,0.3333,0.2857,2
2,1.0000,1.0,0.0,0.0,0.1760,0.7000,0.3333,0.5000,3
3,0.9983,1.0,0.0,0.0,0.1617,0.6667,0.3750,0.3333,4
4,1.0000,1.0,0.0,0.0,0.0945,0.5556,0.2500,0.4444,5


Calculate average evaluation

In [43]:
numeric_columns = [
    "Semantic Similarity",
    "Lexical Overlap",
    "Originality (%)",
    "Grammar Score",
    "BLEU",
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L"
]

average_results = results_df[
    numeric_columns
].mean()

print("AVERAGE EVALUATION RESULTS")
print("=" * 50)

for metric, value in average_results.items():

    print(
        f"{metric}: {value:.4f}"
    )

AVERAGE EVALUATION RESULTS
Semantic Similarity: 0.9997
Lexical Overlap: 1.0000
Originality (%): 0.0000
Grammar Score: 40.0000
BLEU: 0.1269
ROUGE-1: 0.6971
ROUGE-2: 0.3117
ROUGE-L: 0.4068


Calculate semantic preservation percentage

In [44]:
semantic_preservation = (
    average_results["Semantic Similarity"] * 100
)

print(
    f"Average semantic preservation: "
    f"{semantic_preservation:.2f}%"
)

Average semantic preservation: 99.97%


Calculate average originality

In [45]:
average_originality = average_results[
    "Originality (%)"
]

print(
    f"Average estimated lexical originality: "
    f"{average_originality:.2f}%"
)

Average estimated lexical originality: 0.00%


Generate final evaluation report

In [46]:
def generate_evaluation_report(results_df):

    avg = results_df[
        numeric_columns
    ].mean()

    report = f"""
============================================================
AI-POWERED PARAPHRASING TOOL
EVALUATION REPORT
============================================================

Model:
Vamsi/T5_Paraphrase_Paws

Semantic Similarity:
{avg['Semantic Similarity']:.4f}

Average Lexical Overlap:
{avg['Lexical Overlap']:.4f}

Estimated Lexical Originality:
{avg['Originality (%)']:.2f}%

Grammar Score:
{avg['Grammar Score']:.2f}/100

BLEU Score:
{avg['BLEU']:.4f}

ROUGE-1:
{avg['ROUGE-1']:.4f}

ROUGE-2:
{avg['ROUGE-2']:.4f}

ROUGE-L:
{avg['ROUGE-L']:.4f}

Number of test examples:
{len(results_df)}

============================================================
INTERPRETATION
============================================================

1. Semantic Similarity measures whether the generated
   paraphrase preserves the meaning of the source text.

2. Lexical Originality measures how much the vocabulary
   differs from the original text.

3. Grammar Score measures detected grammar/spelling/style
   issues using LanguageTool.

4. BLEU measures similarity between the generated text
   and a human/reference paraphrase.

5. ROUGE measures lexical overlap between the generated
   output and reference paraphrase.

6. High semantic similarity combined with reasonable
   lexical originality indicates effective paraphrasing.

7. BLEU and ROUGE should not be interpreted alone because
   valid paraphrases may use substantially different wording.

============================================================
LIMITATIONS
============================================================

- Generated text may occasionally change the original meaning.
- T5 generation quality depends on the input text.
- LanguageTool does not guarantee perfect grammar.
- Lexical originality is not equivalent to plagiarism detection.
- BLEU and ROUGE require a reference text for meaningful
  reference-based evaluation.
- Human evaluation is recommended for final validation.

============================================================
"""

    return report

Diplay final report

In [47]:
report = generate_evaluation_report(
    results_df
)

print(report)


AI-POWERED PARAPHRASING TOOL
EVALUATION REPORT

Model:
Vamsi/T5_Paraphrase_Paws

Semantic Similarity:
0.9997

Average Lexical Overlap:
1.0000

Estimated Lexical Originality:
0.00%

Grammar Score:
40.00/100

BLEU Score:
0.1269

ROUGE-1:
0.6971

ROUGE-2:
0.3117

ROUGE-L:
0.4068

Number of test examples:
5

INTERPRETATION

1. Semantic Similarity measures whether the generated
   paraphrase preserves the meaning of the source text.

2. Lexical Originality measures how much the vocabulary
   differs from the original text.

3. Grammar Score measures detected grammar/spelling/style
   issues using LanguageTool.

4. BLEU measures similarity between the generated text
   and a human/reference paraphrase.

5. ROUGE measures lexical overlap between the generated
   output and reference paraphrase.

6. High semantic similarity combined with reasonable
   lexical originality indicates effective paraphrasing.

7. BLEU and ROUGE should not be interpreted alone because
   valid paraphrases may use s

Save evaluation report

In [48]:
with open(
    "evaluation_report.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(report)

print(
    "Evaluation report saved as evaluation_report.txt"
)

Evaluation report saved as evaluation_report.txt


Save sample results

In [49]:
results_df.to_csv(
    "paraphrasing_evaluation_results.csv",
    index=False
)

print(
    "Results saved as paraphrasing_evaluation_results.csv"
)

Results saved as paraphrasing_evaluation_results.csv


Save the model output

In [50]:
with open(
    "sample_output.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(
        "ORIGINAL TEXT\n"
    )

    file.write(
        text + "\n\n"
    )

    file.write(
        "PARAPHRASED TEXT\n"
    )

    file.write(
        final_text
    )

print(
    "Sample output saved successfully."
)

Sample output saved successfully.


Final console application

In [51]:
def run_paraphrasing_application():

    print("\n")
    print("=" * 70)
    print("          AI-POWERED PARAPHRASING TOOL")
    print("=" * 70)

    print("\nEnter your text below.")
    print("Press ENTER twice to finish.\n")

    lines = []

    while True:

        line = input()

        if line.strip() == "":
            break

        lines.append(line)

    if not lines:

        print("\nNo text entered.")

        return

    input_text = " ".join(lines)

    input_text = preprocess_text(
        input_text
    )

    print("\nGenerating paraphrase...")

    output_text = paraphrase_text(
        input_text
    )

    print("\n")
    print("=" * 70)
    print("ORIGINAL TEXT")
    print("=" * 70)

    print(input_text)

    print("\n")
    print("=" * 70)
    print("PARAPHRASED TEXT")
    print("=" * 70)

    print(output_text)

    print("\n")
    print("=" * 70)
    print("QUALITY EVALUATION")
    print("=" * 70)

    evaluation = evaluate_paraphrase(
        input_text,
        output_text
    )

    for metric, value in evaluation.items():

        print(
            f"{metric}: {value}"
        )

    print("\nProcessing completed.")


# Run application
run_paraphrasing_application()



          AI-POWERED PARAPHRASING TOOL

Enter your text below.
Press ENTER twice to finish.



 Database performance is important for applications that process large amounts of information. Poorly written queries, missing indexes, and inefficient database design can increase response time and affect the overall performance of an application.
 



Generating paraphrase...


ORIGINAL TEXT
Database performance is important for applications that process large amounts of information. Poorly written queries, missing indexes, and inefficient database design can increase response time and affect the overall performance of an application.


PARAPHRASED TEXT
Database performance is important for applications that process large amounts of information. Poorly written queries, missing indexes and inefficient database design can increase response time and affect the overall performance of an application .


QUALITY EVALUATION
Semantic Similarity: 0.9991
Lexical Overlap: 1.0
Originality (%): 0.0
Grammar Score: 69.7

Processing completed.


Close Language Tool

In [52]:
if grammar_tool is not None:
    grammar_tool.close()
    print("LanguageTool closed.")

LanguageTool closed.


Conclusion

The developed system demonstrates how Transformer-based deep learning can be used to automatically paraphrase natural-language text. The T5 model generates alternative sentence structures, while semantic similarity, grammar checking, lexical originality, BLEU, and ROUGE provide multiple perspectives for evaluating the generated output. The combination of these techniques provides a practical AI-based paraphrasing pipeline suitable for console-based NLP applications.